# 02. Multi-Agent Handoffs

This notebook shows two related patterns: agents as tools, and real handoffs. The first is useful when a coordinator wants to call a specialist like a function. The second is useful when the specialist should take over the run.

```text
raw document
   |
   v
coordinator
   |-- tool --> OCR cleaner
   |-- tool --> metadata specialist
   |-- tool --> summarization agent
   `-- handoff --> metadata specialist (final take over)
```

## Why this pattern matters
For beginners, the big intuition is that one model does not need to do everything. In humanities workflows, different stages often require different behaviors: careful transcription, factual extraction, and interpretive writing. Separating those responsibilities makes the pipeline easier to debug and easier to explain.

This also helps show that orchestration is not magic. It is a sequence of explicit roles with clear inputs and outputs.

## Concepts
- Agents as tools
- Handoffs
- Pipeline design
- Separation of concerns
- Traceable intermediate outputs

The official docs on [Agents as tools](https://openai.github.io/openai-agents-python/tools/#agents-as-tools) and [Handoffs](https://openai.github.io/openai-agents-python/handoffs/) are especially relevant here.

## Tracing
View traces in the OpenAI Traces dashboard: https://platform.openai.com/traces

## Dataset
Any `.txt` files stored in `data/` are loaded directly in the notebook.

## Colab data setup
If the notebook is opened in Google Colab without cloning the repo, upload `data.zip` from the repository root to the current working directory using the Files panel, then rerun the setup cell below.

## Further reading
- Agents as tools: https://openai.github.io/openai-agents-python/tools/#agents-as-tools
- Handoffs: https://openai.github.io/openai-agents-python/handoffs/
- Sessions: https://openai.github.io/openai-agents-python/sessions/


In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from agents import Agent, Runner, handoff, trace, set_tracing_export_api_key

DEFAULT_MODEL = 'gpt-5.4-mini'

In [ ]:
candidate_dirs = [Path.cwd() / 'data', Path.cwd().parent / 'data']
data_dir = next((path for path in candidate_dirs if path.exists()), None)
if data_dir is None:
    zip_candidates = [Path.cwd() / 'data.zip', Path.cwd().parent / 'data.zip']
    zip_path = next((path for path in zip_candidates if path.exists()), None)
    if zip_path is not None:
        import zipfile
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(zip_path.parent)
        data_dir = next((path for path in candidate_dirs if path.exists()), None)
if data_dir is None:
    if 'google.colab' in sys.modules:
        print('Colab detected, but data/ is still missing.')
        print('Upload data.zip from the repository root into the Files panel, then rerun this cell.')
    else:
        raise FileNotFoundError('data/ folder not found. Clone the repo locally or place data.zip next to the notebook.')

load_dotenv(Path.cwd() / '.env')
load_dotenv(Path.cwd().parent / '.env')
assert os.getenv('OPENAI_API_KEY'), 'Set OPENAI_API_KEY in .env or your environment before running this notebook.'

set_tracing_export_api_key(os.environ['OPENAI_API_KEY'])

def load_documents() -> list[dict[str, str]]:
    documents = []
    for index, path in enumerate(sorted(data_dir.glob('*.txt')), start=1):
        documents.append({
            'id': index,
            'title': path.stem.replace('_', ' ').strip().title(),
            'filename': path.name,
            'text': path.read_text(encoding='utf-8').strip(),
        })
    if not documents:
        raise FileNotFoundError('No .txt files found in data/. Add one or more documents and rerun the notebook.')
    return documents

DOCUMENTS = load_documents()
raw_text = DOCUMENTS[0]['text']
print(raw_text)


Madrid, 4 April 1871

Dear brother,

I have finally found a calm hour to write after the commotion of the exhibition. The galleries were crowded, and several visitors asked about the catalog, which made the day long but worthwhile.

In 1871, Maria Gomez wrote from Madrid to her brother in Valencia about the exhibition and the costs of travel. She also said she hoped the spring weather would make the journey easier when she next visited.

With affection,
Maria


## Step 1: Define the specialist agents

The OCR cleaner focuses on transcription quality. The metadata specialist focuses on structured facts. The summarization agent focuses on prose.

The key teaching point is that each agent receives a narrower instructions block. Narrower agents are easier to test because a failure usually means a single stage needs adjustment.

In [3]:
ocr_cleaner = Agent(
    name='OCR Cleaner',
    instructions=(
        'Fix obvious OCR mistakes while preserving original meaning. '
        'Do not invent new information.'
    ),
    model=DEFAULT_MODEL,
)

metadata_specialist = Agent(
    name='Metadata Specialist',
    instructions='Extract people, places, dates, and key facts from cleaned historical text.',
    model=DEFAULT_MODEL,
)

summarizer = Agent(
    name='Summarization Agent',
    instructions='Write a two-sentence summary grounded in the provided cleaned text and extracted facts.',
    model=DEFAULT_MODEL,
)

ocr_cleaner, metadata_specialist, summarizer

(Agent(name='OCR Cleaner', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions='Fix obvious OCR mistakes while preserving original meaning. Do not invent new information.', prompt=None, handoffs=[], model='gpt-5.4-mini', model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=Reasoning(effort='none', generate_summary=None, summary=None), verbosity='low', metadata=None, store=None, prompt_cache_retention=None, include_usage=None, response_include=None, top_logprobs=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None, retry=None, context_management=None), input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True),
 Agent(name='Metadata Specialist', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions='

## Step 2: Use agents as tools

The coordinator can call other agents with `as_tool`. That is often the easiest way to teach agents-as-tools because it feels like ordinary function calling, but the function happens to be another agent.

Conceptually, this is useful when you want one agent to remain in charge while delegating a subtask to a specialist. The coordinator does not need to know the specialist's internals, only its purpose.

In [4]:
cleaner_tool = ocr_cleaner.as_tool('clean_ocr', 'Clean OCR text while preserving meaning.')
extractor_tool = metadata_specialist.as_tool('extract_metadata', 'Extract structured metadata from cleaned text.')
summarizer_tool = summarizer.as_tool('summarize_history', 'Write a short summary based on cleaned text and facts.')

print(cleaner_tool.name, extractor_tool.name, summarizer_tool.name)

clean_ocr extract_metadata summarize_history


## Step 3: Real handoff

A handoff transfers control to another agent. Unlike `as_tool`, the next agent becomes the active agent for the rest of the run. That makes handoffs the right choice when a specialist should take ownership of the task rather than act as a callable subroutine.

In this example, the coordinator can call all three specialists as tools, and then hand off to the metadata specialist when it should take over the run.

In [6]:
handoff_metadata_specialist = Agent(
    name='Metadata Specialist',
    instructions=(
        'You receive cleaned OCR text from the coordinator. Extract people, places, dates, and key facts. '
        'Return concise evidence-based metadata.'
    ),
    model=DEFAULT_MODEL,
)

handoff_coordinator = Agent(
    name='Coordinator',
    instructions=(
        'Clean obvious OCR errors first. Then hand off to the metadata specialist. '
        'Make conservative edits and keep evidence visible.'
    ),
    tools=[cleaner_tool, extractor_tool, summarizer_tool],
    handoffs=[handoff(handoff_metadata_specialist, tool_name_override='transfer_to_metadata_specialist')],
    model=DEFAULT_MODEL,
)

handoff_metadata_specialist, handoff_coordinator

(Agent(name='Metadata Specialist', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions='You receive cleaned OCR text from the coordinator. Extract people, places, dates, and key facts. Return concise evidence-based metadata.', prompt=None, handoffs=[], model='gpt-5.4-mini', model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=Reasoning(effort='none', generate_summary=None, summary=None), verbosity='low', metadata=None, store=None, prompt_cache_retention=None, include_usage=None, response_include=None, top_logprobs=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None, retry=None, context_management=None), input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True),
 Agent(name='Coordinator', handoff_description=None, tools=[Fu

## Step 4: Run the handoff pipeline

This call is fully runnable once the API key is set. In notebooks, use `await Runner.run(...)` rather than `run_sync(...)` because the kernel already manages an event loop. The coordinator can use the tool agents first, and the final result should come from the specialist agent that received the handoff.

If you want to connect this to the SDK docs, this is the place to introduce tracing and the idea that each delegation can be followed in the run history.

In [7]:
with trace('DH handoff pipeline'):
    result = await Runner.run(handoff_coordinator, raw_text)
print(result.final_output)


{
  "people": [
    {
      "name": "Maria Gomez",
      "role": "writer",
      "evidence": "In 1871, Maria Gomez wrote from Madrid to her brother in Valencia..."
    },
    {
      "name": "brother",
      "role": "recipient",
      "evidence": "Dear brother"
    }
  ],
  "places": [
    {
      "name": "Madrid",
      "role": "origin",
      "evidence": "Madrid, 4 April 1871"
    },
    {
      "name": "Valencia",
      "role": "destination",
      "evidence": "wrote from Madrid to her brother in Valencia"
    }
  ],
  "dates": [
    {
      "date": "4 April 1871",
      "evidence": "Madrid, 4 April 1871"
    },
    {
      "date": "1871",
      "evidence": "In 1871, Maria Gomez wrote..."
    }
  ],
  "key_facts": [
    "Letter written from Madrid on 4 April 1871.",
    "Maria Gomez mentioned the exhibition, crowded galleries, and visitors asking about the catalog.",
    "She discussed the costs of travel.",
    "She hoped spring weather would make her next journey easier."
  ]
}


## Further reading
## Exercise

Add a fourth agent that classifies whether the text looks like a letter, notice, or newspaper excerpt.

## Solution

Create `classifier = Agent(...)`, wrap it with `classifier.as_tool(...)`, add it to `handoff_coordinator.tools`, and update the instructions to call it before the handoff.